# Load Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM

MODEL_ID = "roneneldan/TinyStories-3M"
MODEL_REVISION = "cfaf26ec85ecdfc1bd7c2638104cce55cb67f894"

model_fp32 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float32,
)

model_fp32 = model_fp32.cpu()
model_fp32.eval()

print("Model class:", type(model_fp32).__name__)
print("Parameter dtype:", next(model_fp32.parameters()).dtype)
print("Device:", next(model_fp32.parameters()).device)

/home/randy/Documents/ACompArc/model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1011 21:16:42.905000 14901 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1011 21:16:42.927000 14901 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 108/108 [00:00<00:00, 56559.47it/s]
[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-3M
Key               

Model class: GPTNeoForCausalLM
Parameter dtype: torch.float32
Device: cpu


# Test Model

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

prompt = "Once upon a time, a little girl found a tiny dragon in her garden."

inputs = tokenizer(
    prompt,
    add_special_tokens=False,
    return_tensors="pt",
).to(model_fp32.device)

with torch.no_grad():
    generated_ids = model_fp32.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        num_beams=1,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print("Input token count:", inputs["input_ids"].shape[1])
print("\nStory:")
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

Input token count: 16

Story:
Once upon a time, a little girl found a tiny dragon in her garden. She was so excited and wanted to show it to her mom.

The little girl ran to her mom and said, "Mommy, look! A dragon!"

Her mom smiled and said, "Yes, it's very big and strong. But be careful, it's very dangerous."

The little girl nodded and said, "I will be careful. I will be careful


# TinyStories-3M Quantization — Fixed Specification

## Model

| Setting | Value |
|---|---|
| Hugging Face model | `roneneldan/TinyStories-3M` |
| Revision | `cfaf26ec85ecdfc1bd7c2638104cce55cb67f894` |
| Architecture | GPT-Neo |
| Transformer blocks | 8 |
| Hidden size | 128 |
| Attention heads | 16 |
| Head dimension | 8 |
| MLP dimensions | 128 → 512 → 128 |
| Reference model | Pretrained FP32 model, CPU, evaluation mode |

## Input and Generation

| Setting | Value |
|---|---|
| Batch size | 1 |
| Input sequence length | 32 tokens |
| Tokenization | Matching tokenizer; `add_special_tokens=False` |
| Input selection | First 32 token IDs; prompt must contain at least 32 tokens |
| Position IDs | 0–31 |
| Block-0 input | Token embeddings + position embeddings; dropout inactive |
| Block input/output shape | `[1, 32, 128]` |
| Generation limit | 20 additional tokens; may stop earlier at EOS |
| Decoding | Greedy: `do_sample=False`, `num_beams=1` |
| Generation cache | Enabled |
| Maximum resulting sequence length | 52 tokens |


In [3]:
# Input specification
BATCH_SIZE = 1
SEQUENCE_LENGTH = 32
GENERATED_LENGTH = 20

PROMPT = (
    "Once upon a time, a curious little girl named Lily lived in a quiet "
    "village beside a green forest. Every morning, she carried fresh bread "
    "to her grandmother and watched the birds in the trees."
)

# Verify the loaded model's architecture.
config = model_fp32.config

assert config.num_layers == 8
assert config.hidden_size == 128
assert config.num_heads == 16

HEAD_DIM = config.hidden_size // config.num_heads
MLP_HIDDEN_SIZE = config.intermediate_size or 4 * config.hidden_size

assert HEAD_DIM == 8
assert MLP_HIDDEN_SIZE == 512

BLOCK_INPUT_SHAPE = (
    BATCH_SIZE,
    SEQUENCE_LENGTH,
    config.hidden_size,
)

# Choices to apply when we construct the quantizer.
QUANTIZATION_SETTINGS = {
    "scheme": "W8A8",
    "is_per_channel": False,
    "is_dynamic": False,
    "is_qat": False,
}

# Used later for full-model generation tests.
GENERATION_SETTINGS = {
    "max_new_tokens": GENERATED_LENGTH,
    "do_sample": False,
    "num_beams": 1,
    "use_cache": True,
}

print("Block input shape:", BLOCK_INPUT_SHAPE)
print("Head dimension:", HEAD_DIM)
print("MLP hidden size:", MLP_HIDDEN_SIZE)
print("Quantization settings:", QUANTIZATION_SETTINGS)

Block input shape: (1, 32, 128)
Head dimension: 8
MLP hidden size: 512
Quantization settings: {'scheme': 'W8A8', 'is_per_channel': False, 'is_dynamic': False, 'is_qat': False}


# Saving FP32 Model From TinyStories - 3M


In [ ]:
from pathlib import Path
import json

FP32_MODEL_DIR = (
    Path.home()
    / "Documents/ACompArc/model/project/artifacts/models/tinystories_3m_fp32"
)

# Check that the model parameters are still FP32.
assert all(p.dtype == torch.float32 for p in model_fp32.parameters())

# Preserve any previously saved copy.
FP32_MODEL_DIR.mkdir(parents=True, exist_ok=False)

model_fp32.save_pretrained(FP32_MODEL_DIR)
tokenizer.save_pretrained(FP32_MODEL_DIR)

metadata = {
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "dtype": "float32",
    "quantized": False,
}

(FP32_MODEL_DIR / "source_metadata.json").write_text(
    json.dumps(metadata, indent=2) + "\n"
)

print("Saved FP32 model and tokenizer to:")
print(FP32_MODEL_DIR)

print("\nSaved files:")
for path in sorted(FP32_MODEL_DIR.iterdir()):
    print(path.name)

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 14.90it/s]

Saved FP32 model and tokenizer to:
/home/randy/Documents/ACompArc/model/project/artifacts/models/tinystories_3m_fp32

Saved files:
config.json
generation_config.json
model.safetensors
source_metadata.json
tokenizer.json
tokenizer_config.json


In [ ]:
# LOAD IT
# from pathlib import Path
# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer

# FP32_MODEL_DIR = Path(
#     "/home/randy/Documents/ACompArc/model/project/"
#     "artifacts/models/tinystories_3m_fp32"
# )

# # Load the saved FP32 model.
# model_fp32 = AutoModelForCausalLM.from_pretrained(
#     FP32_MODEL_DIR,
#     local_files_only=True,
#     dtype=torch.float32,
# )
# model_fp32 = model_fp32.cpu()
# model_fp32.eval()

# # Load its matching tokenizer.
# tokenizer = AutoTokenizer.from_pretrained(
#     FP32_MODEL_DIR,
#     local_files_only=True,
# )

# print("Model:", type(model_fp32).__name__)
# print("Weight dtype:", next(model_fp32.parameters()).dtype)
# print("Device:", next(model_fp32.parameters()).device)
# print("Transformer blocks:", len(model_fp32.transformer.h))

# Divide each model per block

In [ ]:
import copy

model_fp32.eval()

assert len(model_fp32.transformer.h) == 8

# Independent copies, indexed from block 0 through block 7.
blocks_fp32 = [
    copy.deepcopy(block).cpu().eval()
    for block in model_fp32.transformer.h
]

# Convenient name for the block we will study first.
block0_fp32 = blocks_fp32[0]

print("Extracted Transformer blocks:", len(blocks_fp32))

for i, block in enumerate(blocks_fp32):
    parameter_count = sum(p.numel() for p in block.parameters())
    print(f"Block {i}: {parameter_count:,} parameters")

print("\nComponents inside block 0:")
for name, component in block0_fp32.named_children():
    print(f"  {name}: {type(component).__name__}")

Extracted Transformer blocks: 8
Block 0: 197,888 parameters
Block 1: 197,888 parameters
Block 2: 197,888 parameters
Block 3: 197,888 parameters
Block 4: 197,888 parameters
Block 5: 197,888 parameters
Block 6: 197,888 parameters
Block 7: 197,888 parameters

Components inside block 0:
  ln_1: LayerNorm
  attn: GPTNeoAttention
  ln_2: LayerNorm
  mlp: GPTNeoMLP


In [6]:
print(block0_fp32)

GPTNeoBlock(
  (ln_1): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (attn): GPTNeoAttention(
    (attention): GPTNeoSelfAttention(
      (attn_dropout): Dropout(p=0.0, inplace=False)
      (resid_dropout): Dropout(p=0.0, inplace=False)
      (k_proj): Linear(in_features=128, out_features=128, bias=False)
      (v_proj): Linear(in_features=128, out_features=128, bias=False)
      (q_proj): Linear(in_features=128, out_features=128, bias=False)
      (out_proj): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (ln_2): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (mlp): GPTNeoMLP(
    (c_fc): Linear(in_features=128, out_features=512, bias=True)
    (c_proj): Linear(in_features=512, out_features=128, bias=True)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.0, inplace=False)
  )
)


# Quantization Block 0 - 7 

## Preparation

In [9]:
import copy
import torch
import numpy as np
from torch import nn

# Working copy for export; the original model stays available.
export_model_fp32 = copy.deepcopy(model_fp32).cpu().eval()
export_model_fp32.set_attn_implementation("eager")

assert len(export_model_fp32.transformer.h) == 8
assert BATCH_SIZE == 1
assert SEQUENCE_LENGTH == 32

def collect_block_io(input_ids):
    """Capture the actual FP32 input and output of every block."""
    captured = {}
    handles = []

    def make_hook(block_index):
        def hook(module, args, output):
            hidden_output = (
                output[0] if isinstance(output, (tuple, list)) else output
            )
            captured[block_index] = (
                args[0].detach().clone(),
                hidden_output.detach().clone(),
            )
        return hook

    try:
        for i, block in enumerate(export_model_fp32.transformer.h):
            handles.append(block.register_forward_hook(make_hook(i)))

        with torch.no_grad():
            export_model_fp32.transformer(
                input_ids=input_ids,
                use_cache=False,
                return_dict=True,
            )
    finally:
        for handle in handles:
            handle.remove()

    assert len(captured) == 8
    return [captured[i] for i in range(8)]

# Fixed benchmark prompt from our specification cell.
benchmark_ids = tokenizer(
    PROMPT,
    add_special_tokens=False,
    return_tensors="pt",
)["input_ids"]

assert benchmark_ids.shape[1] >= SEQUENCE_LENGTH
benchmark_ids = benchmark_ids[:, :SEQUENCE_LENGTH]
benchmark_block_io = collect_block_io(benchmark_ids)

# Small initial calibration corpus.
calibration_prompts = [
    "Once upon a time, a little girl named Lily lived beside a quiet "
    "forest. Every morning she carried bread to her grandmother and "
    "greeted all the animals along the path.",

    "Tom found a small red ball underneath his bed. He took it outside "
    "and played with his dog until the sun disappeared behind the hills.",

    "A young rabbit wanted to cross the river, but the water was too "
    "deep. His friends gathered branches and helped him build a bridge.",

    "Mia planted a tiny seed in the garden. She watered it every day, "
    "and soon a tall sunflower grew beside her bedroom window.",

    "The little bird could not fly as high as the others. She practiced "
    "every afternoon until she finally reached the top of the old tree.",

    "Ben heard a strange sound inside the kitchen. He opened the door "
    "slowly and discovered a hungry kitten looking for some food.",

    "A kind farmer shared his vegetables with the people in his village. "
    "When winter arrived, everyone helped him repair his wooden barn.",

    "Sara lost her favorite blue hat at the park. She searched near the "
    "swings and found it beside a sleeping brown puppy.",
]

# Concatenate stories with EOS separators, then take full 32-token chunks.
assert tokenizer.eos_token_id is not None
calibration_tokens = []

for text in calibration_prompts:
    calibration_tokens.extend(
        tokenizer.encode(text, add_special_tokens=False)
    )
    calibration_tokens.append(tokenizer.eos_token_id)

calibration_ids = [
    torch.tensor(
        [calibration_tokens[start:start + SEQUENCE_LENGTH]],
        dtype=torch.long,
    )
    for start in range(
        0,
        len(calibration_tokens) - SEQUENCE_LENGTH + 1,
        SEQUENCE_LENGTH,
    )
]

assert calibration_ids
calibration_block_inputs = [[] for _ in range(8)]

for ids in calibration_ids:
    block_io = collect_block_io(ids)
    for i, (block_input, _) in enumerate(block_io):
        calibration_block_inputs[i].append(block_input)

print("Calibration sequences:", len(calibration_ids))

for i, (block_input, _) in enumerate(benchmark_block_io):
    assert tuple(block_input.shape) == (1, 32, 128)
    print(
        f"Block {i}: input {tuple(block_input.shape)}, "
        f"{len(calibration_block_inputs[i])} calibration samples"
    )

Calibration sequences: 7
Block 0: input (1, 32, 128), 7 calibration samples
Block 1: input (1, 32, 128), 7 calibration samples
Block 2: input (1, 32, 128), 7 calibration samples
Block 3: input (1, 32, 128), 7 calibration samples
Block 4: input (1, 32, 128), 7 calibration samples
Block 5: input (1, 32, 128), 7 calibration samples
Block 6: input (1, 32, 128), 7 calibration samples
Block 7: input (1, 32, 128), 7 calibration samples


## Quantize, Validate, Save

In [10]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import importlib.metadata as package_metadata
import hashlib
import json

import litert_torch
from ai_edge_litert.interpreter import Interpreter, OpResolverType
from torchao.quantization.pt2e.quantize_pt2e import (
    prepare_pt2e,
    convert_pt2e,
)
from litert_torch.quantize.pt2e_quantizer import (
    PT2EQuantizer,
    get_symmetric_quantization_config,
)
from litert_torch.quantize.quant_config import QuantConfig


class BlockExportWrapper(nn.Module):
    def __init__(self, block):
        super().__init__()
        self.block = block

    def forward(self, hidden_states):
        output = self.block(
            hidden_states,
            layer_past=None,
            attention_mask=None,
            use_cache=False,
            output_attentions=False,
        )
        return output[0] if isinstance(output, (tuple, list)) else output


def as_numpy(output):
    if isinstance(output, (tuple, list)):
        output = output[0]
    if isinstance(output, torch.Tensor):
        output = output.detach().cpu().numpy()
    return np.asarray(output)


run_name = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")

QUANTIZED_BLOCK_DIR = (
    Path.home()
    / "Documents/ACompArc/model/project/artifacts/models"
    / "tinystories_3m_blocks_w8a8"
    / run_name
)
QUANTIZED_BLOCK_DIR.mkdir(parents=True, exist_ok=False)

report = {
    "model_id": MODEL_ID,
    "revision": MODEL_REVISION,
    "sequence_length": SEQUENCE_LENGTH,
    "use_cache": False,
    "attention_implementation": "eager",
    "quantization": {
        "is_per_channel": False,
        "is_dynamic": False,
        "is_qat": False,
        "fold_quantize": False,
        "fully_integer": False,
    },
    "calibration_prompts": calibration_prompts,
    "calibration_sequences": len(calibration_ids),
    "calibration_source": "preceding FP32 blocks",
    "benchmark_prompt": PROMPT,
    "versions": {
        name: package_metadata.version(name)
        for name in [
            "torch", "transformers", "torchao",
            "litert-torch", "ai-edge-litert", "numpy",
        ]
    },
    "blocks": [],
}

# Preserve the token sequences and per-block benchmark references.
validation_data = {
    "benchmark_input_ids": benchmark_ids.numpy(),
    "calibration_input_ids": torch.cat(calibration_ids).numpy(),
}
for i, (x, y) in enumerate(benchmark_block_io):
    validation_data[f"block{i}_input_fp32"] = x.numpy()
    validation_data[f"block{i}_output_fp32"] = y.numpy()

np.savez(
    QUANTIZED_BLOCK_DIR / "validation_inputs.npz",
    **validation_data,
)

print("Saving to:", QUANTIZED_BLOCK_DIR, flush=True)

for i in range(8):
    print(f"\nQuantizing block {i}...", flush=True)

    wrapper = BlockExportWrapper(
        copy.deepcopy(export_model_fp32.transformer.h[i])
    ).eval()

    example_input, native_reference = benchmark_block_io[i]
    sample_args = (example_input,)

    # Check that isolated execution matches the block in the full model.
    with torch.no_grad():
        isolated_reference = wrapper(example_input)

    torch.testing.assert_close(
        isolated_reference,
        native_reference,
        atol=1e-5,
        rtol=1e-5,
    )

    exported = torch.export.export(wrapper, sample_args)
    with torch.no_grad():
        torch.testing.assert_close(
            exported.module()(example_input),
            isolated_reference,
        )

    quantizer = PT2EQuantizer().set_global(
        get_symmetric_quantization_config(
            is_per_channel=False,
            is_dynamic=False,
            is_qat=False,
        )
    )

    prepared = prepare_pt2e(exported.module(), quantizer)

    with torch.no_grad():
        for calibration_input in calibration_block_inputs[i]:
            prepared(calibration_input)

    quantized = convert_pt2e(prepared, fold_quantize=False)

    edge_block = litert_torch.convert(
        quantized,
        sample_args,
        quant_config=QuantConfig(pt2e_quantizer=quantizer),
    )

    output_before_save = as_numpy(edge_block(example_input))
    model_path = QUANTIZED_BLOCK_DIR / f"block{i}_w8a8.tflite"
    edge_block.export(str(model_path))

    # Verify that the saved file reloads with the same output.
    reloaded = litert_torch.load(str(model_path))
    output_after_save = as_numpy(reloaded(example_input))

    np.testing.assert_allclose(
        output_after_save,
        output_before_save,
        atol=1e-6,
        rtol=1e-6,
    )

    error = np.abs(
        output_after_save - native_reference.numpy()
    )
    assert np.isfinite(output_after_save).all()

    # Inspect the exported tensor types and projection quantization.
    interpreter = Interpreter(
        model_path=str(model_path),
        experimental_op_resolver_type=OpResolverType.BUILTIN_REF,
    )
    interpreter.allocate_tensors()
    tensors = {
        t["index"]: t for t in interpreter.get_tensor_details()
    }
    dtype_counts = dict(Counter(
        np.dtype(t["dtype"]).name for t in tensors.values()
    ))

    int8_fc_count = 0
    for op in interpreter._get_ops_details():
        if op["op_name"] == "FULLY_CONNECTED":
            indices = [
                int(op["inputs"][0]),
                int(op["inputs"][1]),
                int(op["outputs"][0]),
            ]
            if all(
                tensors[index]["dtype"] == np.int8
                for index in indices
            ):
                int8_fc_count += 1

    assert int8_fc_count == 6, (
        f"Block {i}: expected six INT8 projections; "
        f"found {int8_fc_count}. Inspect this export."
    )

    block_report = {
        "block_index": i,
        "file": model_path.name,
        "sha256": hashlib.sha256(model_path.read_bytes()).hexdigest(),
        "bytes": model_path.stat().st_size,
        "int8_fully_connected_ops": int8_fc_count,
        "tensor_dtype_counts": dtype_counts,
        "benchmark_max_abs_error": float(error.max()),
        "benchmark_mean_abs_error": float(error.mean()),
        "reload_check": "passed",
    }
    report["blocks"].append(block_report)

    (QUANTIZED_BLOCK_DIR / "export_report.json").write_text(
        json.dumps(report, indent=2) + "\n"
    )

    print(
        f"Saved {model_path.name} | INT8 FCs: {int8_fc_count} | "
        f"max error: {error.max():.6f} | "
        f"mean error: {error.mean():.6f}",
        flush=True,
    )
    print("Tensor types:", dtype_counts, flush=True)

print("\nSaved all eight blocks to:")
print(QUANTIZED_BLOCK_DIR)

I0000 00:00:1791726593.759456   14901 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Saving to: /home/randy/Documents/ACompArc/model/project/artifacts/models/tinystories_3m_blocks_w8a8/20261011T134954617091Z

Quantizing block 0...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


... (217 outputs left out)

(00:00) [START] Write Model to Bytes

I0000 00:00:1791726608.124172   14901 flatbuffer_export.cc:4342] Estimated count of arithmetic ops: 14.844 M  ops, equivalently 7.422 M  MACs


(00:00) [ DONE] Write Model to Bytes (+00:00)

Saved block7_w8a8.tflite | INT8 FCs: 6 | max error: 0.011993 | mean error: 0.002239
Tensor types: {'float32': 64, 'int32': 22, 'bool': 1, 'int8': 37}

Saved all eight blocks to:
/home/randy/Documents/ACompArc/model/project/artifacts/models/tinystories_3m_blocks_w8a8/20261011T134954617091Z


# Check quantization

## Block 0

In [12]:
from pathlib import Path
from collections import Counter
import numpy as np
from IPython.display import display, Markdown
from ai_edge_litert.interpreter import Interpreter, OpResolverType

BLOCK_EXPORT_DIR = Path(
    "/home/randy/Documents/ACompArc/model/project/artifacts/models/"
    "tinystories_3m_blocks_w8a8/20261011T134954617091Z"
)

block0_interpreter = Interpreter(
    model_path=str(BLOCK_EXPORT_DIR / "block0_w8a8.tflite"),
    experimental_preserve_all_tensors=True,
    experimental_op_resolver_type=OpResolverType.BUILTIN_REF,
)
block0_interpreter.allocate_tensors()

block0_tensors = {
    int(t["index"]): t
    for t in block0_interpreter.get_tensor_details()
}
block0_ops = block0_interpreter._get_ops_details()

shape_ops = {
    "RESHAPE", "TRANSPOSE", "SQUEEZE", "EXPAND_DIMS",
    "SLICE", "STRIDED_SLICE", "GATHER", "GATHER_ND",
    "SPLIT", "SPLIT_V", "PACK", "UNPACK",
    "CONCATENATION", "PAD", "PADV2", "TILE", "SHAPE",
}

def valid_tensors(indices):
    # Negative indices indicate absent optional inputs, such as bias.
    return [
        block0_tensors[int(index)]
        for index in indices
        if int(index) >= 0
    ]

def tensor_labels(tensors):
    return ", ".join(
        f"T{t['index']} {np.dtype(t['dtype']).name}"
        for t in tensors
    ) or "—"

def classify_operation(op):
    name = op["op_name"]
    tensors = (
        valid_tensors(op["inputs"])
        + valid_tensors(op["outputs"])
    )

    if name == "QUANTIZE":
        return "Quantize / requantize"
    if name == "DEQUANTIZE":
        return "Dequantize"
    if name in shape_ops:
        return "Data movement / shape"
    if any(np.issubdtype(t["dtype"], np.floating) for t in tensors):
        return "Floating-point / mixed"
    if any(t["dtype"] == np.int8 for t in tensors):
        return "INT8 computation"
    return "Other integer / boolean"

block0_op_status = [
    classify_operation(op) for op in block0_ops
]

print("Block 0 operation count:", len(block0_ops))
print("\nOperation summary:")
for status, count in Counter(block0_op_status).items():
    print(f"  {status}: {count}")

rows = [
    "| Op | Operation | Inputs | Outputs | Classification |",
    "|---:|---|---|---|---|",
]

for op, status in zip(block0_ops, block0_op_status):
    rows.append(
        f"| {op['index']} | {op['op_name']} | "
        f"{tensor_labels(valid_tensors(op['inputs']))} | "
        f"{tensor_labels(valid_tensors(op['outputs']))} | "
        f"{status} |"
    )

display(Markdown("\n".join(rows)))

Block 0 operation count: 74

Operation summary:
  Floating-point / mixed: 31
  Data movement / shape: 20
  Quantize / requantize: 5
  INT8 computation: 11
  Dequantize: 7


| Op | Operation | Inputs | Outputs | Classification |
|---:|---|---|---|---|
| 0 | MEAN | T0 float32, T10 int32 | T31 float32 | Floating-point / mixed |
| 1 | RESHAPE | T31 float32, T11 int32 | T32 float32 | Data movement / shape |
| 2 | SQUARED_DIFFERENCE | T0 float32, T32 float32 | T33 float32 | Floating-point / mixed |
| 3 | MEAN | T33 float32, T10 int32 | T34 float32 | Floating-point / mixed |
| 4 | ADD | T34 float32, T2 float32 | T35 float32 | Floating-point / mixed |
| 5 | RSQRT | T35 float32 | T36 float32 | Floating-point / mixed |
| 6 | MUL | T36 float32, T31 float32 | T37 float32 | Floating-point / mixed |
| 7 | SUB | T5 float32, T37 float32 | T38 float32 | Floating-point / mixed |
| 8 | RESHAPE | T36 float32, T11 int32 | T39 float32 | Data movement / shape |
| 9 | MUL | T0 float32, T39 float32 | T40 float32 | Floating-point / mixed |
| 10 | RESHAPE | T38 float32, T11 int32 | T41 float32 | Data movement / shape |
| 11 | ADD | T40 float32, T41 float32 | T42 float32 | Floating-point / mixed |
| 12 | MUL | T42 float32, T6 float32 | T43 float32 | Floating-point / mixed |
| 13 | ADD | T43 float32, T7 float32 | T44 float32 | Floating-point / mixed |
| 14 | QUANTIZE | T44 float32 | T45 int8 | Quantize / requantize |
| 15 | FULLY_CONNECTED | T45 int8, T27 int8 | T46 int8 | INT8 computation |
| 16 | FULLY_CONNECTED | T45 int8, T25 int8 | T47 int8 | INT8 computation |
| 17 | FULLY_CONNECTED | T45 int8, T26 int8 | T48 int8 | INT8 computation |
| 18 | RESHAPE | T46 int8, T13 int32 | T49 int8 | Data movement / shape |
| 19 | TRANSPOSE | T49 int8, T14 int32 | T50 int8 | Data movement / shape |
| 20 | RESHAPE | T47 int8, T13 int32 | T51 int8 | Data movement / shape |
| 21 | RESHAPE | T48 int8, T13 int32 | T52 int8 | Data movement / shape |
| 22 | TRANSPOSE | T52 int8, T14 int32 | T53 int8 | Data movement / shape |
| 23 | TRANSPOSE | T51 int8, T1 int32 | T54 int8 | Data movement / shape |
| 24 | RESHAPE | T50 int8, T15 int32 | T55 int8 | Data movement / shape |
| 25 | DEQUANTIZE | T55 int8 | T56 float32 | Dequantize |
| 26 | RESHAPE | T54 int8, T16 int32 | T57 int8 | Data movement / shape |
| 27 | DEQUANTIZE | T57 int8 | T58 float32 | Dequantize |
| 28 | BATCH_MATMUL | T56 float32, T58 float32 | T59 float32 | Floating-point / mixed |
| 29 | RESHAPE | T59 float32, T17 int32 | T60 float32 | Data movement / shape |
| 30 | SELECT_V2 | T21 bool, T60 float32, T18 float32 | T61 float32 | Floating-point / mixed |
| 31 | SOFTMAX | T61 float32 | T62 float32 | Floating-point / mixed |
| 32 | RESHAPE | T62 float32, T19 int32 | T63 float32 | Data movement / shape |
| 33 | RESHAPE | T53 int8, T15 int32 | T64 int8 | Data movement / shape |
| 34 | DEQUANTIZE | T64 int8 | T65 float32 | Dequantize |
| 35 | BATCH_MATMUL | T63 float32, T65 float32 | T66 float32 | Floating-point / mixed |
| 36 | QUANTIZE | T66 float32 | T67 int8 | Quantize / requantize |
| 37 | RESHAPE | T67 int8, T20 int32 | T68 int8 | Data movement / shape |
| 38 | TRANSPOSE | T68 int8, T14 int32 | T69 int8 | Data movement / shape |
| 39 | RESHAPE | T69 int8, T12 int32 | T70 int8 | Data movement / shape |
| 40 | FULLY_CONNECTED | T70 int8, T28 int8, T24 int32 | T71 int8 | INT8 computation |
| 41 | DEQUANTIZE | T71 int8 | T72 float32 | Dequantize |
| 42 | ADD | T72 float32, T0 float32 | T73 float32 | Floating-point / mixed |
| 43 | MEAN | T73 float32, T10 int32 | T74 float32 | Floating-point / mixed |
| 44 | RESHAPE | T74 float32, T11 int32 | T75 float32 | Data movement / shape |
| 45 | SQUARED_DIFFERENCE | T73 float32, T75 float32 | T76 float32 | Floating-point / mixed |
| 46 | MEAN | T76 float32, T10 int32 | T77 float32 | Floating-point / mixed |
| 47 | ADD | T77 float32, T2 float32 | T78 float32 | Floating-point / mixed |
| 48 | RSQRT | T78 float32 | T79 float32 | Floating-point / mixed |
| 49 | MUL | T79 float32, T74 float32 | T80 float32 | Floating-point / mixed |
| 50 | SUB | T5 float32, T80 float32 | T81 float32 | Floating-point / mixed |
| 51 | RESHAPE | T79 float32, T11 int32 | T82 float32 | Data movement / shape |
| 52 | MUL | T73 float32, T82 float32 | T83 float32 | Floating-point / mixed |
| 53 | RESHAPE | T81 float32, T11 int32 | T84 float32 | Data movement / shape |
| 54 | ADD | T83 float32, T84 float32 | T85 float32 | Floating-point / mixed |
| 55 | MUL | T85 float32, T8 float32 | T86 float32 | Floating-point / mixed |
| 56 | ADD | T86 float32, T9 float32 | T87 float32 | Floating-point / mixed |
| 57 | QUANTIZE | T87 float32 | T88 int8 | Quantize / requantize |
| 58 | FULLY_CONNECTED | T88 int8, T29 int8, T23 int32 | T89 int8 | INT8 computation |
| 59 | DEQUANTIZE | T89 int8 | T90 float32 | Dequantize |
| 60 | MUL | T89 int8, T91 int8 | T92 int8 | INT8 computation |
| 61 | POW | T90 float32, T3 float32 | T93 float32 | Floating-point / mixed |
| 62 | QUANTIZE | T93 float32 | T94 int8 | Quantize / requantize |
| 63 | MUL | T94 int8, T95 int8 | T96 int8 | INT8 computation |
| 64 | ADD | T89 int8, T96 int8 | T97 int8 | INT8 computation |
| 65 | MUL | T97 int8, T98 int8 | T99 int8 | INT8 computation |
| 66 | DEQUANTIZE | T99 int8 | T100 float32 | Dequantize |
| 67 | TANH | T100 float32 | T101 float32 | Floating-point / mixed |
| 68 | ADD | T101 float32, T4 float32 | T102 float32 | Floating-point / mixed |
| 69 | QUANTIZE | T102 float32 | T103 int8 | Quantize / requantize |
| 70 | MUL | T92 int8, T103 int8 | T104 int8 | INT8 computation |
| 71 | FULLY_CONNECTED | T104 int8, T30 int8, T22 int32 | T105 int8 | INT8 computation |
| 72 | DEQUANTIZE | T105 int8 | T106 float32 | Dequantize |
| 73 | ADD | T73 float32, T106 float32 | T107 float32 | Floating-point / mixed |

In [8]:
from pathlib import Path
import hashlib
import numpy as np
import torch
from transformers import AutoModelForCausalLM
from ai_edge_litert.interpreter import Interpreter, OpResolverType

PROJECT_DIR = Path(
    "/home/randy/Documents/ACompArc/model/project"
)
BLOCK_DIR = PROJECT_DIR / (
    "artifacts/models/tinystories_3m_blocks_w8a8/"
    "20261011T134954617091Z"
)
BLOCK0_PATH = BLOCK_DIR / "block0_w8a8.tflite"
FP32_DIR = PROJECT_DIR / "artifacts/models/tinystories_3m_fp32"

# Read the checkpoint that produced our saved benchmark inputs.
inspection_model = AutoModelForCausalLM.from_pretrained(
    FP32_DIR,
    local_files_only=True,
    dtype=torch.float32,
).cpu().eval()

with np.load(BLOCK_DIR / "validation_inputs.npz") as saved:
    benchmark_ids = saved["benchmark_input_ids"].copy()
    benchmark_x = saved["block0_input_fp32"].copy()
    calibration_ids = saved["calibration_input_ids"].copy()

def embedding_input(ids):
    ids = torch.as_tensor(ids, dtype=torch.long).reshape(1, 32)
    positions = torch.arange(32, dtype=torch.long).unsqueeze(0)

    with torch.no_grad():
        x = inspection_model.transformer.drop(
            inspection_model.transformer.wte(ids)
            + inspection_model.transformer.wpe(positions)
        )

    return x.numpy().copy()

# Confirm the saved checkpoint reproduces the benchmark input.
np.testing.assert_allclose(
    embedding_input(benchmark_ids),
    benchmark_x,
    atol=1e-6,
    rtol=1e-6,
)

interpreter = Interpreter(
    model_path=str(BLOCK0_PATH),
    experimental_preserve_all_tensors=True,
    experimental_op_resolver_type=OpResolverType.BUILTIN_REF,
)
interpreter.allocate_tensors()

op0 = interpreter._get_ops_details()[0]
assert op0["op_name"] == "MEAN"

x_index = int(op0["inputs"][0])
axes_index = int(op0["inputs"][1])
mean_index = int(op0["outputs"][0])
axes = tuple(
    int(axis) for axis in interpreter.get_tensor(axes_index).reshape(-1)
)

# Retain these cases in memory for the next integer-reference cell.
op0_cases = []
samples = [("benchmark", benchmark_ids, benchmark_x)]

for i, ids in enumerate(calibration_ids):
    samples.append((f"calibration_{i}", ids, embedding_input(ids)))

for name, ids, x in samples:
    assert x.shape == (1, 32, 128)
    assert np.isfinite(x).all()

    interpreter.set_tensor(x_index, x)
    interpreter.invoke()

    mean_ref = interpreter.get_tensor(mean_index).copy()
    mean_python = np.mean(
        x, axis=axes, dtype=np.float32
    ).reshape(mean_ref.shape)

    op0_cases.append({
        "name": name,
        "input_ids": np.asarray(ids).reshape(1, 32).copy(),
        "x_fp32": x.copy(),
        "mean_ref_fp32": mean_ref,
        "mean_python_fp32": mean_python,
    })

def print_range(label, cases):
    x = np.concatenate([c["x_fp32"].reshape(-1) for c in cases])
    means = np.concatenate([
        c["mean_ref_fp32"].reshape(-1) for c in cases
    ])
    max_error = max(
        float(np.max(np.abs(
            c["mean_ref_fp32"] - c["mean_python_fp32"]
        )))
        for c in cases
    )

    print(f"\n{label}")
    print("  Sequences:", len(cases))
    print("  Input values:", x.size)
    print(f"  T0 minimum: {x.min():.9f}")
    print(f"  T0 maximum: {x.max():.9f}")
    print(f"  T0 maximum absolute value: {np.abs(x).max():.9f}")
    print(f"  Mean minimum: {means.min():.9f}")
    print(f"  Mean maximum: {means.max():.9f}")
    print(f"  Python vs LiteRT mean max error: {max_error:.9e}")

op0_source_sha256 = hashlib.sha256(BLOCK0_PATH.read_bytes()).hexdigest()

print("Source:", BLOCK0_PATH)
print("SHA-256:", op0_source_sha256)
print(f"Op 0: MEAN(T{x_index}, T{axes_index}) -> T{mean_index}")
print("Reduction axes:", axes)
print("Input shape:", benchmark_x.shape)
print("Output shape:", op0_cases[0]["mean_ref_fp32"].shape)

print_range("Benchmark input", op0_cases[:1])
print_range("Calibration inputs", op0_cases[1:])

print("\nFirst token, benchmark:")
print("  First 8 T0 features:", benchmark_x[0, 0, :8])
print("  Reference mean:",
      op0_cases[0]["mean_ref_fp32"].reshape(-1)[0])

Loading weights: 100%|██████████| 108/108 [00:00<00:00, 14202.38it/s]

Source: /home/randy/Documents/ACompArc/model/project/artifacts/models/tinystories_3m_blocks_w8a8/20261011T134954617091Z/block0_w8a8.tflite
SHA-256: 06481f6c5151a83e3c01896d314d902ee9608773d94834848777ccc7282d87a9
Op 0: MEAN(T0, T10) -> T31
Reduction axes: (0, 2)
Input shape: (1, 32, 128)
Output shape: (32,)

Benchmark input
  Sequences: 1
  Input values: 4096
  T0 minimum: -0.328789324
  T0 maximum: 0.307286024
  T0 maximum absolute value: 0.328789324
  Mean minimum: -0.009422798
  Mean maximum: 0.004446551
  Python vs LiteRT mean max error: 3.492459655e-09

Calibration inputs
  Sequences: 7
  Input values: 28672
  T0 minimum: -0.469448209
  T0 maximum: 0.455547750
  T0 maximum absolute value: 0.469448209
  Mean minimum: -0.009570063
  Mean maximum: 0.007038387
  Python vs LiteRT mean max error: 6.053596735e-09

First token, benchmark:
  First 8 T0 features: [-0.08634469 -0.03653849  0.14335527  0.0147367   0.02153916 -0.04537516
 -0.10777003 -0.2002483 ]
  Reference mean: -0.000785323

In [9]:
import numpy as np

T0_FRAC_BITS = 8
T0_SCALE = 2.0 ** (-T0_FRAC_BITS)

FEATURE_COUNT = 128
T31_SCALE = T0_SCALE / FEATURE_COUNT  # 1 / 32768

# Keep trial results in memory.
op0_integer_trial = []

for case in op0_cases:
    x = case["x_fp32"]  # T0 (block input)
    mean_ref = case["mean_ref_fp32"].reshape(32)  # T31 (FP32 means)

    # Quantize T0: nearest integer, ties to even.
    rounded = np.rint(x.astype(np.float64) / T0_SCALE)
    clipped_count = int(np.count_nonzero(
        (rounded < -128) | (rounded > 127)
    ))
    t0_q = np.clip(rounded, -128, 127).astype(np.int8)

    # Integer Op 0:
    # Store the sum as T31 with scale T0_SCALE / 128.
    t31_mean_q = np.sum(
        t0_q.astype(np.int32),
        axis=-1,
        dtype=np.int32,
    ).reshape(32)

    # Reconstruct represented values for accuracy comparisons only.
    t0_approx = t0_q.astype(np.float64) * T0_SCALE
    t31_approx = t31_mean_q.astype(np.float64) * T31_SCALE

    op0_integer_trial.append({
        "name": case["name"],
        "t0_q": t0_q,
        "t31_mean_q": t31_mean_q,
        "t31_approx": t31_approx,
        "clipped_count": clipped_count,
        "input_error": np.abs(t0_approx - x),
        "mean_error": np.abs(t31_approx - mean_ref),
    })

def report_trial(label, trials):
    input_errors = np.concatenate([
        t["input_error"].reshape(-1) for t in trials
    ])
    mean_errors = np.concatenate([
        t["mean_error"] for t in trials
    ])
    q_values = np.concatenate([
        t["t0_q"].reshape(-1) for t in trials
    ])
    mean_q_values = np.concatenate([
        t["t31_mean_q"] for t in trials
    ])

    print(f"\n{label}")
    print("  T0 (input) clipped values:",
          sum(t["clipped_count"] for t in trials))
    print("  T0 (input) integer range:",
          int(q_values.min()), "to", int(q_values.max()))
    print(f"  T0 (input) max absolute error: {input_errors.max():.9e}")
    print(f"  T0 (input) mean absolute error: {input_errors.mean():.9e}")
    print("  T31 (means) integer range:",
          int(mean_q_values.min()), "to", int(mean_q_values.max()))
    print(f"  T31 (means) max absolute error: {mean_errors.max():.9e}")
    print(f"  T31 (means) mean absolute error: {mean_errors.mean():.9e}")

print("Candidate format:")
print("  T0 (input): INT8, scale =", T0_SCALE, ", zero point = 0")
print("  T31 (means): INT32, scale =", T31_SCALE, ", zero point = 0")

report_trial("Benchmark", op0_integer_trial[:1])
report_trial("Calibration", op0_integer_trial[1:])

first = op0_integer_trial[0]
first_ref = op0_cases[0]["mean_ref_fp32"].reshape(32)[0]

print("\nFirst benchmark token:")
print("  T0 (input), first 8 integers:", first["t0_q"][0, 0, :8])
print("  T31 (mean), stored integer:", first["t31_mean_q"][0])
print(f"  T31 (mean), represented value: {first['t31_approx'][0]:.9f}")
print(f"  T31 (mean), FP32 reference: {first_ref:.9f}")
print(f"  T31 (mean), absolute error: {first['mean_error'][0]:.9e}")

Candidate format:
  T0 (input): INT8, scale = 0.00390625 , zero point = 0
  T31 (means): INT32, scale = 3.0517578125e-05 , zero point = 0

Benchmark
  T0 (input) clipped values: 0
  T0 (input) integer range: -84 to 79
  T0 (input) max absolute error: 1.952394843e-03
  T0 (input) mean absolute error: 9.877213568e-04
  T31 (means) integer range: -306 to 144
  T31 (means) max absolute error: 2.882166300e-04
  T31 (means) mean absolute error: 4.977694425e-05

Calibration
  T0 (input) clipped values: 0
  T0 (input) integer range: -120 to 117
  T0 (input) max absolute error: 1.953020692e-03
  T0 (input) mean absolute error: 9.742461435e-04
  T31 (means) integer range: -317 to 232
  T31 (means) max absolute error: 2.882166300e-04
  T31 (means) mean absolute error: 7.735267893e-05

First benchmark token:
  T0 (input), first 8 integers: [-22  -9  37   4   6 -12 -28 -51]
  T31 (mean), stored integer: -25
  T31 (mean), represented value: -0.000762939
  T31 (mean), FP32 reference: -0.000785323
  T

In [12]:
import numpy as np

# Explicit format definitions.
T32_FRAC_BITS = T0_FRAC_BITS + 7  # 8 + log2(128) = 15
T32_SCALE = 2.0 ** (-T32_FRAC_BITS)

T33_FINE_FRAC_BITS = 2 * T32_FRAC_BITS  # Squaring: 30
T33_FINE_SCALE = 2.0 ** (-T33_FINE_FRAC_BITS)

# Candidate coarser format; not a final choice.
T33_COARSE_FRAC_BITS = 15
T33_COARSE_SCALE = 2.0 ** (-T33_COARSE_FRAC_BITS)

ALIGN_SHIFT = T32_FRAC_BITS - T0_FRAC_BITS       # 7
REQUANT_SHIFT = T33_FINE_FRAC_BITS - T33_COARSE_FRAC_BITS  # 15

np.testing.assert_equal(T32_SCALE, T31_SCALE)

# Identify T33 (squared differences) in the reference graph.
op2 = interpreter._get_ops_details()[2]
assert op2["op_name"] == "SQUARED_DIFFERENCE"
t33_index = int(op2["outputs"][0])

op2_integer_trial = []

for case, mean_trial in zip(op0_cases, op0_integer_trial):
    assert case["name"] == mean_trial["name"]

    # Read T33 (FP32 squared differences) from the saved TFLite.
    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()
    t33_ref = interpreter.get_tensor(t33_index).copy()

    # Op 1: T31 (means) -> T32 (broadcast means).
    # Same integers and scale; only the shape changes.
    t32_q = mean_trial["t31_mean_q"].astype(np.int64).reshape(1, 32, 1)

    # Op 2: align T0 (input) to T32's scale.
    # INT64 temporaries ensure multiplication/squaring is performed widely.
    t0_aligned_q = (
        mean_trial["t0_q"].astype(np.int64) * (1 << ALIGN_SHIFT)
    )
    difference_q = t0_aligned_q - t32_q
    t33_fine_q = difference_q * difference_q

    # Requantize nonnegative squared values.
    # Round to nearest; halfway cases round upward.
    rounding_offset = 1 << (REQUANT_SHIFT - 1)
    t33_coarse_q = (
        t33_fine_q + rounding_offset
    ) >> REQUANT_SHIFT

    assert t33_coarse_q.max() <= np.iinfo(np.int32).max
    t33_coarse_q = t33_coarse_q.astype(np.int32)

    # Represented real values: diagnostics only.
    fine_approx = t33_fine_q.astype(np.float64) * T33_FINE_SCALE
    coarse_approx = t33_coarse_q.astype(np.float64) * T33_COARSE_SCALE

    op2_integer_trial.append({
        "name": case["name"],
        "t32_q": t32_q,
        "t0_aligned_q": t0_aligned_q,
        "difference_q": difference_q,
        "t33_ref": t33_ref,
        "t33_fine_q": t33_fine_q,
        "t33_coarse_q": t33_coarse_q,
        "fine_approx": fine_approx,
        "coarse_approx": coarse_approx,
        "fine_error": np.abs(fine_approx - t33_ref),
        "coarse_error": np.abs(coarse_approx - t33_ref),
        "requant_error": np.abs(coarse_approx - fine_approx),
    })

def report_squared_trial(label, trials):
    print(f"\n{label}")

    for key, description in [
        ("fine_error", "T33 at scale 2^-30 vs FP32"),
        ("coarse_error", "T33 at scale 2^-15 vs FP32"),
        ("requant_error", "Additional change from requantization"),
    ]:
        errors = np.concatenate([t[key].reshape(-1) for t in trials])
        print(f"  {description}:")
        print(f"    Max absolute error:  {errors.max():.9e}")
        print(f"    Mean absolute error: {errors.mean():.9e}")

    for key, description in [
        ("t33_fine_q", "T33 (fine squared integers)"),
        ("t33_coarse_q", "T33 (coarse squared integers)"),
    ]:
        largest = max(int(t[key].max()) for t in trials)

        # Range check for a later sum over 128 features.
        largest_sum = max(
            int(np.sum(t[key], axis=-1, dtype=np.int64).max())
            for t in trials
        )
        print(f"  {description}: max = {largest}")
        print(f"    Largest 128-feature sum: {largest_sum}")
        print(f"    Sum fits INT32: {largest_sum <= np.iinfo(np.int32).max}")

print("T32 (broadcast means): scale =", T32_SCALE)
print("T33 (fine squared differences): scale =", T33_FINE_SCALE)
print("T33 (coarse squared differences): scale =", T33_COARSE_SCALE)

report_squared_trial("Benchmark", op2_integer_trial[:1])
report_squared_trial("Calibration", op2_integer_trial[1:])

first = op2_integer_trial[0]
position = (0, 0, 0)

print("\nFirst token, first feature:")
print("  T0 (aligned input) integer:", first["t0_aligned_q"][position])
print("  T32 (broadcast mean) integer:", first["t32_q"][position])
print("  Centered difference integer:", first["difference_q"][position])
print("  T33 (fine squared difference) integer:", first["t33_fine_q"][position])
print("  T33 (coarse squared difference) integer:", first["t33_coarse_q"][position])
print(f"  T33 (fine represented value): {first['fine_approx'][position]:.9f}")
print(f"  T33 (coarse represented value): {first['coarse_approx'][position]:.9f}")
print(f"  T33 (FP32 reference): {first['t33_ref'][position]:.9f}")

T32 (broadcast means): scale = 3.0517578125e-05
T33 (fine squared differences): scale = 9.313225746154785e-10
T33 (coarse squared differences): scale = 3.0517578125e-05

Benchmark
  T33 at scale 2^-30 vs FP32:
    Max absolute error:  1.116245985e-03
    Mean absolute error: 1.534825375e-04
  T33 at scale 2^-15 vs FP32:
    Max absolute error:  1.101002097e-03
    Mean absolute error: 1.537359188e-04
  Additional change from requantization:
    Max absolute error:  1.525506377e-05
    Mean absolute error: 7.570248272e-06
  T33 (fine squared integers): max = 118330884
    Largest 128-feature sum: 2239203200
    Sum fits INT32: False
  T33 (coarse squared integers): max = 3611
    Largest 128-feature sum: 68342
    Sum fits INT32: True

Calibration
  T33 at scale 2^-30 vs FP32:
    Max absolute error:  1.424428076e-03
    Mean absolute error: 1.494486447e-04
  T33 at scale 2^-15 vs FP32:
    Max absolute error:  1.412481070e-03
    Mean absolute error: 1.497123103e-04
  Additional change

In [13]:
# Op 3: MEAN
# T33 (squared differences) -> T34 (per-token variance)

op3 = interpreter._get_ops_details()[3]
assert op3["op_name"] == "MEAN"
assert int(op3["inputs"][0]) == t33_index

t34_index = int(op3["outputs"][0])
variance_axes = tuple(
    int(v)
    for v in interpreter.get_tensor(int(op3["inputs"][1])).reshape(-1)
)
assert variance_axes == (0, 2)

# Division by 128 is represented through the output scale.
T34_FINE_FRAC_BITS = T33_FINE_FRAC_BITS + 7
T34_COARSE_FRAC_BITS = T33_COARSE_FRAC_BITS + 7

T34_FINE_SCALE = T33_FINE_SCALE / FEATURE_COUNT
T34_COARSE_SCALE = T33_COARSE_SCALE / FEATURE_COUNT

int32_max = int(np.iinfo(np.int32).max)
op3_integer_trial = []

assert len(op0_cases) == len(op2_integer_trial)

for case, squared_trial in zip(op0_cases, op2_integer_trial):
    assert case["name"] == squared_trial["name"]

    # Reference: run the original mixed-precision TFLite graph.
    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()
    t34_ref = interpreter.get_tensor(t34_index).copy().reshape(-1)

    # Fine path: sum T33 at scale 2^-30 using INT64.
    t34_fine_q = np.sum(
        squared_trial["t33_fine_q"],
        axis=variance_axes,
        dtype=np.int64,
    )

    # Check the coarse sum in INT64 before using INT32.
    coarse_sum_check = np.sum(
        squared_trial["t33_coarse_q"],
        axis=variance_axes,
        dtype=np.int64,
    )
    assert coarse_sum_check.min() >= 0
    assert coarse_sum_check.max() <= int32_max

    # Coarse path: sum T33 at scale 2^-15 using INT32.
    t34_coarse_q = np.sum(
        squared_trial["t33_coarse_q"],
        axis=variance_axes,
        dtype=np.int32,
    )

    # Floating-point reconstruction is only for diagnostics.
    fine_approx = t34_fine_q.astype(np.float64) * T34_FINE_SCALE
    coarse_approx = t34_coarse_q.astype(np.float64) * T34_COARSE_SCALE

    op3_integer_trial.append({
        "name": case["name"],
        "t34_ref": t34_ref,
        "t34_fine_q": t34_fine_q,
        "t34_coarse_q": t34_coarse_q,
        "fine_approx": fine_approx,
        "coarse_approx": coarse_approx,
        "fine_error": np.abs(fine_approx - t34_ref),
        "coarse_error": np.abs(coarse_approx - t34_ref),
        "requant_error": np.abs(coarse_approx - fine_approx),
    })


def report_variance_trial(label, trials):
    print(f"\n{label}")

    reference = np.concatenate([t["t34_ref"] for t in trials])
    print(
        "  T34 (FP32 variance) range:",
        f"{reference.min():.9e} to {reference.max():.9e}",
    )

    for key, description in [
        ("fine_error", "Fine variance vs FP32"),
        ("coarse_error", "Coarse variance vs FP32"),
        ("requant_error", "Variance change from T33 requantization"),
    ]:
        errors = np.concatenate([t[key] for t in trials])
        print(f"  {description}:")
        print(f"    Max absolute error:  {errors.max():.9e}")
        print(f"    Mean absolute error: {errors.mean():.9e}")

    for key, description in [
        ("t34_fine_q", "T34 (fine variance), INT64"),
        ("t34_coarse_q", "T34 (coarse variance), INT32"),
    ]:
        values = np.concatenate([t[key] for t in trials])
        print(f"  {description}: {values.min()} to {values.max()}")


print(
    "T34 (fine variance): INT64,",
    f"fractional bits = {T34_FINE_FRAC_BITS}, scale = {T34_FINE_SCALE}",
)
print(
    "T34 (coarse variance): INT32,",
    f"fractional bits = {T34_COARSE_FRAC_BITS}, scale = {T34_COARSE_SCALE}",
)

report_variance_trial("Benchmark", op3_integer_trial[:1])
report_variance_trial("Calibration", op3_integer_trial[1:])

first = op3_integer_trial[0]
print("\nFirst benchmark token:")
print("  T34 (fine variance), stored integer:", first["t34_fine_q"][0])
print("  T34 (coarse variance), stored integer:", first["t34_coarse_q"][0])
print("  T34 (fine represented value):", f'{first["fine_approx"][0]:.9f}')
print("  T34 (coarse represented value):", f'{first["coarse_approx"][0]:.9f}')
print("  T34 (FP32 reference):", f'{first["t34_ref"][0]:.9f}')

T34 (fine variance): INT64, fractional bits = 37, scale = 7.275957614183426e-12
T34 (coarse variance): INT32, fractional bits = 22, scale = 2.384185791015625e-07

Benchmark
  T34 (FP32 variance) range: 6.802664604e-03 to 1.629877090e-02
  Fine variance vs FP32:
    Max absolute error:  4.531163722e-05
    Mean absolute error: 1.453356526e-05
  Coarse variance vs FP32:
    Max absolute error:  4.650373012e-05
    Mean absolute error: 1.458609768e-05
  Variance change from T33 requantization:
    Max absolute error:  3.814697266e-06
    Mean absolute error: 8.474162314e-07
  T34 (fine variance), INT64: 936912768 to 2239203200
  T34 (coarse variance), INT32: 28593 to 68342

Calibration
  T34 (FP32 variance) range: 6.579582114e-03 to 1.848954707e-02
  Fine variance vs FP32:
    Max absolute error:  7.481500506e-05
    Mean absolute error: 1.634376947e-05
  Coarse variance vs FP32:
    Max absolute error:  7.554609329e-05
    Mean absolute error: 1.619344403e-05
  Variance change from T33 r

### OP4

In [14]:
# Op 4: ADD
# T34 (variance) + T2 (epsilon) -> T35 (variance plus epsilon)
# Continue with the coarse INT32 variance candidate.

op4 = interpreter._get_ops_details()[4]
assert op4["op_name"] == "ADD"
assert int(op4["inputs"][0]) == t34_index

epsilon_index = int(op4["inputs"][1])
t35_index = int(op4["outputs"][0])

# Read epsilon from this specific TFLite model.
epsilon_tensor = interpreter.get_tensor(epsilon_index)
assert epsilon_tensor.size == 1
epsilon_fp32 = float(epsilon_tensor.reshape(-1)[0])
assert np.isfinite(epsilon_fp32) and epsilon_fp32 > 0

T35_FRAC_BITS = T34_COARSE_FRAC_BITS
T35_SCALE = T34_COARSE_SCALE

# Offline constant conversion: nearest rounding, ties to even.
epsilon_q = int(np.rint(epsilon_fp32 / T35_SCALE))
epsilon_approx = epsilon_q * T35_SCALE

int32_limits = np.iinfo(np.int32)
assert int32_limits.min <= epsilon_q <= int32_limits.max

print(f"T{epsilon_index} (epsilon), FP32: {epsilon_fp32:.12e}")
print(f"T{epsilon_index} (epsilon), stored integer: {epsilon_q}")
print(f"T{epsilon_index} (epsilon), represented: {epsilon_approx:.12e}")
print(
    "Epsilon quantization absolute error:",
    f"{abs(epsilon_approx - epsilon_fp32):.12e}",
)
print(
    f"T{t35_index} (variance plus epsilon): INT32, "
    f"fractional bits = {T35_FRAC_BITS}, scale = {T35_SCALE}"
)
print("Output requantization shift: 0")

op4_integer_trial = []
assert len(op0_cases) == len(op3_integer_trial)

for case, variance_trial in zip(op0_cases, op3_integer_trial):
    assert case["name"] == variance_trial["name"]

    # Actual FP32 T35 from the original TFLite graph.
    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()
    t35_ref = interpreter.get_tensor(t35_index).copy().reshape(-1)

    t34_q = variance_trial["t34_coarse_q"]

    # Check addition in INT64 before performing the INT32 addition.
    sum_check = t34_q.astype(np.int64) + epsilon_q
    assert sum_check.min() >= int32_limits.min
    assert sum_check.max() <= int32_limits.max

    # Integer operation: same scales, so add directly.
    t35_q = t34_q + np.int32(epsilon_q)
    assert t35_q.dtype == np.int32
    assert np.array_equal(t35_q.astype(np.int64), sum_check)

    # Reconstruction only for numerical diagnostics.
    t35_approx = t35_q.astype(np.float64) * T35_SCALE

    op4_integer_trial.append({
        "name": case["name"],
        "t34_q": t34_q.copy(),
        "t35_q": t35_q,
        "t35_ref": t35_ref,
        "t35_approx": t35_approx,
        "error": np.abs(t35_approx - t35_ref),
    })


def report_epsilon_trial(label, trials):
    reference = np.concatenate([t["t35_ref"] for t in trials])
    integers = np.concatenate([t["t35_q"] for t in trials])
    errors = np.concatenate([t["error"] for t in trials])

    print(f"\n{label}")
    print(
        "  T35 (FP32 variance plus epsilon) range:",
        f"{reference.min():.9e} to {reference.max():.9e}",
    )
    print(
        "  T35 (stored integer) range:",
        f"{integers.min()} to {integers.max()}",
    )
    print(f"  Max absolute error vs FP32:  {errors.max():.9e}")
    print(f"  Mean absolute error vs FP32: {errors.mean():.9e}")
    print("  INT32 overflow: none")


report_epsilon_trial("Benchmark", op4_integer_trial[:1])
report_epsilon_trial("Calibration", op4_integer_trial[1:])

first = op4_integer_trial[0]
print("\nFirst benchmark token:")
print("  T34 (variance), stored integer:", first["t34_q"][0])
print("  T2 (epsilon), stored integer:", epsilon_q)
print("  T35 (variance plus epsilon), stored integer:", first["t35_q"][0])
print("  T35 (represented value):", f'{first["t35_approx"][0]:.9f}')
print("  T35 (FP32 reference):", f'{first["t35_ref"][0]:.9f}')
print("  Absolute error:", f'{first["error"][0]:.9e}')

T2 (epsilon), FP32: 9.999999747379e-06
T2 (epsilon), stored integer: 42
T2 (epsilon), represented: 1.001358032227e-05
Epsilon quantization absolute error: 1.358057488687e-08
T35 (variance plus epsilon): INT32, fractional bits = 22, scale = 2.384185791015625e-07
Output requantization shift: 0

Benchmark
  T35 (FP32 variance plus epsilon) range: 6.812664680e-03 to 1.630877145e-02
  T35 (stored integer) range: 28635 to 68384
  Max absolute error vs FP32:  4.651769996e-05
  Mean absolute error vs FP32: 1.458522456e-05
  INT32 overflow: none

Calibration
  T35 (FP32 variance plus epsilon) range: 6.589582190e-03 to 1.849954762e-02
  T35 (stored integer) range: 27611 to 77610
  Max absolute error vs FP32:  7.556006312e-05
  Mean absolute error vs FP32: 1.619393255e-05
  INT32 overflow: none

First benchmark token:
  T34 (variance), stored integer: 68342
  T2 (epsilon), stored integer: 42
  T35 (variance plus epsilon), stored integer: 68384
  T35 (represented value): 0.016304016
  T35 (FP32 re

In [15]:
# Op 5: RSQRT
# T35 (variance plus epsilon) -> T36 (inverse standard deviation)
# Integer implementation: integer division + integer square root.

from math import isqrt

op5 = interpreter._get_ops_details()[5]
assert op5["op_name"] == "RSQRT"
assert int(op5["inputs"][0]) == t35_index

t36_index = int(op5["outputs"][0])

T36_FRAC_BITS = 16
T36_SCALE = 2.0 ** (-T36_FRAC_BITS)

# Derived from input and output scales:
# q_T36 ≈ sqrt(2^(F_input + 2*F_output) / q_T35)
RSQRT_NUMERATOR_BITS = T35_FRAC_BITS + 2 * T36_FRAC_BITS
RSQRT_NUMERATOR = 1 << RSQRT_NUMERATOR_BITS

int64_max = int(np.iinfo(np.int64).max)
int32_max = int(np.iinfo(np.int32).max)

assert RSQRT_NUMERATOR <= int64_max
assert T35_SCALE == 2.0 ** (-T35_FRAC_BITS)

op5_integer_trial = []
assert len(op0_cases) == len(op4_integer_trial)

for case, epsilon_trial in zip(op0_cases, op4_integer_trial):
    assert case["name"] == epsilon_trial["name"]

    # Actual T36 from the original TFLite graph.
    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()
    t36_ref = interpreter.get_tensor(t36_index).copy().reshape(-1)
    assert np.all(np.isfinite(t36_ref)) and np.all(t36_ref > 0)

    t35_q = epsilon_trial["t35_q"].reshape(-1)
    assert np.all(t35_q > 0)

    division_values = []
    output_values = []

    # Integer-only candidate calculation.
    for stored_input in t35_q:
        q = int(stored_input)

        n = RSQRT_NUMERATOR // q
        y = isqrt(n)

        assert 0 <= n <= int64_max
        assert 0 <= y <= int32_max

        # Verify y is exactly floor(sqrt(n)).
        assert y * y <= n < (y + 1) * (y + 1)

        # Verify it is also floor(sqrt(numerator / q)).
        assert y * y * q <= RSQRT_NUMERATOR
        assert RSQRT_NUMERATOR < (y + 1) * (y + 1) * q

        division_values.append(n)
        output_values.append(y)

    division_q = np.array(division_values, dtype=np.int64)
    t36_q = np.array(output_values, dtype=np.int32)

    # Floating-point reconstruction and comparisons: diagnostics only.
    t36_approx = t36_q.astype(np.float64) * T36_SCALE
    represented_t35 = t35_q.astype(np.float64) * T35_SCALE

    # Ideal RSQRT of our represented integer input.
    ideal_from_integer_input = 1.0 / np.sqrt(represented_t35)

    local_error = np.abs(t36_approx - ideal_from_integer_input)
    upstream_error = np.abs(ideal_from_integer_input - t36_ref)
    total_error = np.abs(t36_approx - t36_ref)

    # Allow a tiny tolerance for floating-point diagnostic arithmetic.
    assert np.all(t36_approx <= ideal_from_integer_input + 1e-12)
    assert local_error.max() < T36_SCALE + 1e-12

    op5_integer_trial.append({
        "name": case["name"],
        "t35_q": t35_q.copy(),
        "division_q": division_q,
        "t36_q": t36_q,
        "t36_ref": t36_ref,
        "t36_approx": t36_approx,
        "ideal_from_integer_input": ideal_from_integer_input,
        "local_error": local_error,
        "upstream_error": upstream_error,
        "total_error": total_error,
        "relative_error": total_error / np.abs(t36_ref),
    })


def report_rsqrt_trial(label, trials):
    reference = np.concatenate([t["t36_ref"] for t in trials])
    integers = np.concatenate([t["t36_q"] for t in trials])

    print(f"\n{label}")
    print(
        "  T36 (FP32 inverse standard deviation) range:",
        f"{reference.min():.9f} to {reference.max():.9f}",
    )
    print(
        "  T36 (stored integer) range:",
        f"{integers.min()} to {integers.max()}",
    )

    for key, description in [
        ("local_error", "Op 5's own approximation error"),
        ("upstream_error", "Effect of earlier input/variance errors"),
        ("total_error", "Total error vs TFLite T36"),
    ]:
        errors = np.concatenate([t[key] for t in trials])
        print(f"  {description}:")
        print(f"    Max absolute error:  {errors.max():.9e}")
        print(f"    Mean absolute error: {errors.mean():.9e}")

    relative = np.concatenate([t["relative_error"] for t in trials])
    print(f"  Maximum total relative error: {100 * relative.max():.6f}%")
    print("  Checked intermediate/output widths: pass")


print(f"Integer numerator: 2^{RSQRT_NUMERATOR_BITS}")
print(
    f"T36 (inverse standard deviation): INT32, "
    f"fractional bits = {T36_FRAC_BITS}, scale = {T36_SCALE}"
)
print("Rounding rule: floor")
print(f"Local absolute-error bound: less than {T36_SCALE:.9e}")

report_rsqrt_trial("Benchmark", op5_integer_trial[:1])
report_rsqrt_trial("Calibration", op5_integer_trial[1:])

first = op5_integer_trial[0]
print("\nFirst benchmark token:")
print("  T35 (variance plus epsilon), stored integer:", first["t35_q"][0])
print("  Integer division result:", first["division_q"][0])
print("  T36 (inverse standard deviation), stored integer:", first["t36_q"][0])
print("  T36 (represented value):", f'{first["t36_approx"][0]:.9f}')
print(
    "  Ideal RSQRT of represented T35:",
    f'{first["ideal_from_integer_input"][0]:.9f}',
)
print("  T36 (TFLite FP32 reference):", f'{first["t36_ref"][0]:.9f}')
print("  Op 5 local error:", f'{first["local_error"][0]:.9e}')
print("  Total error vs TFLite:", f'{first["total_error"][0]:.9e}')

Integer numerator: 2^54
T36 (inverse standard deviation): INT32, fractional bits = 16, scale = 1.52587890625e-05
Rounding rule: floor
Local absolute-error bound: less than 1.525878906e-05

Benchmark
  T36 (FP32 inverse standard deviation) range: 7.830498219 to 12.115504265
  T36 (stored integer) range: 513254 to 793160
  Op 5's own approximation error:
    Max absolute error:  1.451589615e-05
    Mean absolute error: 6.953473999e-06
  Effect of earlier input/variance errors:
    Max absolute error:  2.598329202e-02
    Mean absolute error: 8.710513817e-03
  Total error vs TFLite T36:
    Max absolute error:  2.598571777e-02
    Mean absolute error: 8.710637689e-03
  Maximum total relative error: 0.250124%
  Checked intermediate/output widths: pass

Calibration
  T36 (FP32 inverse standard deviation) range: 7.352235794 to 12.318875313
  T36 (stored integer) range: 481782 to 807734
  Op 5's own approximation error:
    Max absolute error:  1.507452264e-05
    Mean absolute error: 7.47512

In [16]:
# Op 6–11: normalization
# T42 = T0 * T36 - T31 * T36
#
# Preserve fine intermediates, then test an INT32 T42 candidate.

ops = interpreter._get_ops_details()
expected_names = {
    6: "MUL",
    7: "SUB",
    8: "RESHAPE",
    9: "MUL",
    10: "RESHAPE",
    11: "ADD",
}
for index, expected in expected_names.items():
    assert ops[index]["op_name"] == expected

stage_indices = {
    index: int(ops[index]["outputs"][0])
    for index in expected_names
}

# Op 7 subtracts the correction from the stored zero constant.
zero_tensor = interpreter.get_tensor(int(ops[7]["inputs"][0]))
assert np.all(zero_tensor == 0)

MEAN_FRAC_BITS = T0_FRAC_BITS + 7
assert T31_SCALE == 2.0 ** (-MEAN_FRAC_BITS)

T37_FRAC_BITS = MEAN_FRAC_BITS + T36_FRAC_BITS
T40_FRAC_BITS = T0_FRAC_BITS + T36_FRAC_BITS
T42_FINE_FRAC_BITS = T37_FRAC_BITS

T42_FRAC_BITS = 16             # Candidate output format.
T42_SCALE = 2.0 ** (-T42_FRAC_BITS)
T42_FINE_SCALE = 2.0 ** (-T42_FINE_FRAC_BITS)

NORMALIZATION_ALIGN_SHIFT = T42_FINE_FRAC_BITS - T40_FRAC_BITS
NORMALIZATION_REQUANT_SHIFT = T42_FINE_FRAC_BITS - T42_FRAC_BITS

assert NORMALIZATION_ALIGN_SHIFT >= 0
assert NORMALIZATION_REQUANT_SHIFT >= 0

stage_descriptions = {
    6: "mean correction",
    7: "negative mean correction",
    8: "broadcast inverse standard deviation",
    9: "scaled input",
    10: "broadcast negative correction",
    11: "normalized features, fine",
}
stage_frac_bits = {
    6: T37_FRAC_BITS,
    7: T37_FRAC_BITS,
    8: T36_FRAC_BITS,
    9: T40_FRAC_BITS,
    10: T37_FRAC_BITS,
    11: T42_FINE_FRAC_BITS,
}

int32_limits = np.iinfo(np.int32)
normalization_integer_trial = []

assert len(op0_cases) == len(op0_integer_trial) == len(op5_integer_trial)

for case, mean_trial, rsqrt_trial in zip(
    op0_cases, op0_integer_trial, op5_integer_trial
):
    assert case["name"] == mean_trial["name"] == rsqrt_trial["name"]

    # Capture all six original TFLite reference tensors.
    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()
    references = {
        index: interpreter.get_tensor(stage_indices[index]).copy()
        for index in expected_names
    }

    # Promote before multiplication.
    input_q = mean_trial["t0_q"].astype(np.int64)
    mean_q = mean_trial["t31_mean_q"].astype(np.int64).reshape(-1)
    inverse_std_q = rsqrt_trial["t36_q"].astype(np.int64).reshape(-1)

    # Op 6: mean * inverse standard deviation; scale 2^-31.
    correction_q = mean_q * inverse_std_q

    # Op 7: 0 - correction; same scale.
    negative_correction_q = -correction_q

    # Op 8: reshape inverse standard deviation; same values/scale.
    broadcast_inverse_std_q = inverse_std_q.reshape(1, -1, 1)

    # Op 9: input * inverse standard deviation; scale 2^-24.
    scaled_input_q = input_q * broadcast_inverse_std_q

    # Op 10: reshape negative correction; scale 2^-31.
    broadcast_negative_q = negative_correction_q.reshape(1, -1, 1)

    # Op 11: align T40 to T41's scale, then add.
    aligned_scaled_input_q = (
        scaled_input_q * (1 << NORMALIZATION_ALIGN_SHIFT)
    )
    t42_fine_q = aligned_scaled_input_q + broadcast_negative_q

    # Verify the equivalent fused expression: (input - mean) * r.
    centered_input_q = (
        input_q * (1 << (MEAN_FRAC_BITS - T0_FRAC_BITS))
        - mean_q.reshape(1, -1, 1)
    )
    fused_q = centered_input_q * broadcast_inverse_std_q
    assert np.array_equal(t42_fine_q, fused_q)

    stage_q = {
        6: correction_q,
        7: negative_correction_q,
        8: broadcast_inverse_std_q,
        9: scaled_input_q,
        10: broadcast_negative_q,
        11: t42_fine_q,
    }

    stage_errors = {}
    for index, integers in stage_q.items():
        reference = references[index]
        assert integers.shape == reference.shape
        scale = 2.0 ** (-stage_frac_bits[index])
        represented = integers.astype(np.float64) * scale
        stage_errors[index] = np.abs(represented - reference)

    # Optional output requantization.
    # NumPy signed >> performs arithmetic right shift.
    # Nearest rounding, ties toward positive infinity.
    shift = NORMALIZATION_REQUANT_SHIFT
    if shift == 0:
        t42_candidate_q = t42_fine_q.copy()
    else:
        offset = 1 << (shift - 1)
        t42_candidate_q = (t42_fine_q + offset) >> shift

    assert t42_candidate_q.min() >= int32_limits.min
    assert t42_candidate_q.max() <= int32_limits.max
    t42_q = t42_candidate_q.astype(np.int32)

    fine_approx = t42_fine_q.astype(np.float64) * T42_FINE_SCALE
    t42_approx = t42_q.astype(np.float64) * T42_SCALE
    requant_error = np.abs(t42_approx - fine_approx)

    if shift > 0:
        assert requant_error.max() <= T42_SCALE / 2 + 1e-12

    normalization_integer_trial.append({
        "name": case["name"],
        "stage_q": stage_q,
        "stage_errors": stage_errors,
        "t42_fine_q": t42_fine_q,
        "t42_q": t42_q,
        "t42_ref": references[11],
        "t42_approx": t42_approx,
        "output_error": np.abs(t42_approx - references[11]),
        "requant_error": requant_error,
        "input_q": input_q,
        "mean_q": mean_q,
        "inverse_std_q": inverse_std_q,
        "centered_input_q": centered_input_q,
    })


def report_normalization(label, trials):
    print(f"\n{label}")
    print("  Intermediate errors vs TFLite:")

    for index in expected_names:
        errors = np.concatenate([
            t["stage_errors"][index].reshape(-1) for t in trials
        ])
        tensor_index = stage_indices[index]
        print(
            f"    Op {index}, T{tensor_index} "
            f"({stage_descriptions[index]}): "
            f"max={errors.max():.9e}, mean={errors.mean():.9e}"
        )

    fine = np.concatenate([t["t42_fine_q"].reshape(-1) for t in trials])
    coarse = np.concatenate([t["t42_q"].reshape(-1) for t in trials])
    reference = np.concatenate([t["t42_ref"].reshape(-1) for t in trials])
    output_error = np.concatenate([
        t["output_error"].reshape(-1) for t in trials
    ])
    requant_error = np.concatenate([
        t["requant_error"].reshape(-1) for t in trials
    ])

    fine_fits_int32 = (
        fine.min() >= int32_limits.min
        and fine.max() <= int32_limits.max
    )

    print(
        "  T42 (FP32 normalized features) range:",
        f"{reference.min():.9f} to {reference.max():.9f}",
    )
    print(f"  T42 (fine integer) range: {fine.min()} to {fine.max()}")
    print(f"  Fine T42 fits INT32: {fine_fits_int32}")
    print(f"  T42 (candidate INT32) range: {coarse.min()} to {coarse.max()}")
    print(f"  Candidate max error vs FP32:  {output_error.max():.9e}")
    print(f"  Candidate mean error vs FP32: {output_error.mean():.9e}")
    print(f"  Additional requantization max change:  {requant_error.max():.9e}")
    print(f"  Additional requantization mean change: {requant_error.mean():.9e}")


print("Op 11 input alignment multiplier:", 1 << NORMALIZATION_ALIGN_SHIFT)
print("T42 (fine normalized features): INT64, scale =", T42_FINE_SCALE)
print("T42 (candidate normalized features): INT32, scale =", T42_SCALE)
print("Output requantization shift:", NORMALIZATION_REQUANT_SHIFT)
print("Rounding: nearest, ties toward positive infinity")

report_normalization("Benchmark", normalization_integer_trial[:1])
report_normalization("Calibration", normalization_integer_trial[1:])

first = normalization_integer_trial[0]
location = (0, 0, 0)

print("\nFirst benchmark token, first feature:")
print("  T0 (input), integer:", first["input_q"][location])
print("  T31 (mean), integer:", first["mean_q"][0])
print("  T36 (inverse standard deviation), integer:", first["inverse_std_q"][0])
print("  Centered input, integer at scale 2^-15:", first["centered_input_q"][location])
print("  T42 (fine normalized feature), integer:", first["t42_fine_q"][location])
print("  T42 (candidate normalized feature), integer:", first["t42_q"][location])
print("  T42 (candidate represented value):", f'{first["t42_approx"][location]:.9f}')
print("  T42 (TFLite FP32 reference):", f'{first["t42_ref"][location]:.9f}')

Op 11 input alignment multiplier: 128
T42 (fine normalized features): INT64, scale = 4.656612873077393e-10
T42 (candidate normalized features): INT32, scale = 1.52587890625e-05
Output requantization shift: 15
Rounding: nearest, ties toward positive infinity

Benchmark
  Intermediate errors vs TFLite:
    Op 6, T37 (mean correction): max=2.653919160e-03, mean=5.205223279e-04
    Op 7, T38 (negative mean correction): max=2.653919160e-03, mean=5.205223279e-04
    Op 8, T39 (broadcast inverse standard deviation): max=2.598571777e-02, mean=8.710637689e-03
    Op 9, T40 (scaled input): max=2.488905191e-02, mean=1.030809522e-02
    Op 10, T41 (broadcast negative correction): max=2.653919160e-03, mean=5.205223279e-04
    Op 11, T42 (normalized features, fine): max=2.518225089e-02, mean=1.027650931e-02
  T42 (FP32 normalized features) range: -3.677406311 to 3.020779133
  T42 (fine integer) range: -7845208188 to 6513916896
  Fine T42 fits INT32: False
  T42 (candidate INT32) range: -239417 to 19

In [17]:
# Inspect Op 12–14:
# T43 = T42 (normalized features) * T6 (gamma)
# T44 = T43 + T7 (beta)
# T45 = quantize(T44), the INT8 projection input

ops = interpreter._get_ops_details()

assert ops[12]["op_name"] == "MUL"
assert ops[13]["op_name"] == "ADD"
assert ops[14]["op_name"] == "QUANTIZE"

t42_index = stage_indices[11]
t43_index = int(ops[12]["outputs"][0])
t44_index = int(ops[13]["outputs"][0])
t45_index = int(ops[14]["outputs"][0])

assert t42_index in ops[12]["inputs"]
assert t43_index in ops[13]["inputs"]
assert int(ops[14]["inputs"][0]) == t44_index

gamma_index = next(
    int(i) for i in ops[12]["inputs"] if int(i) != t42_index
)
beta_index = next(
    int(i) for i in ops[13]["inputs"] if int(i) != t43_index
)

ln1_gamma_fp32 = interpreter.get_tensor(gamma_index).copy()
ln1_beta_fp32 = interpreter.get_tensor(beta_index).copy()

assert ln1_gamma_fp32.size == FEATURE_COUNT
assert ln1_beta_fp32.size == FEATURE_COUNT
assert np.all(np.isfinite(ln1_gamma_fp32))
assert np.all(np.isfinite(ln1_beta_fp32))

details = {
    int(d["index"]): d
    for d in interpreter.get_tensor_details()
}
t45_details = details[t45_index]
assert t45_details["dtype"] == np.int8

quantization = t45_details["quantization_parameters"]
assert len(quantization["scales"]) == 1
assert len(quantization["zero_points"]) == 1

T45_SCALE = float(quantization["scales"][0])
T45_ZERO_POINT = int(quantization["zero_points"][0])
assert T45_SCALE > 0


def report_parameter(index, meaning, values):
    flat = values.reshape(-1)
    print(f"\nT{index} ({meaning})")
    print("  Shape:", values.shape)
    print("  Stored dtype:", values.dtype)
    print(f"  Minimum: {flat.min():.9f}")
    print(f"  Maximum: {flat.max():.9f}")
    print(f"  Maximum absolute value: {np.abs(flat).max():.9f}")
    print("  First 8 values:", flat[:8])


report_parameter(gamma_index, "learned scale gamma", ln1_gamma_fp32)
report_parameter(beta_index, "learned bias beta", ln1_beta_fp32)

print(f"\nT{t45_index} (INT8 projection input)")
print("  Shape:", tuple(t45_details["shape"]))
print("  Scale:", T45_SCALE)
print("  Zero point:", T45_ZERO_POINT)
print(
    "  Representable real range:",
    (-128 - T45_ZERO_POINT) * T45_SCALE,
    "to",
    (127 - T45_ZERO_POINT) * T45_SCALE,
)

# Capture original TFLite outputs for the upcoming integer comparison.
affine_reference_cases = []

for case in op0_cases:
    interpreter.set_tensor(x_index, case["x_fp32"])
    interpreter.invoke()

    affine_reference_cases.append({
        "name": case["name"],
        "t43_ref": interpreter.get_tensor(t43_index).copy(),
        "t44_ref": interpreter.get_tensor(t44_index).copy(),
        "t45_ref": interpreter.get_tensor(t45_index).copy(),
    })


def report_affine_reference(label, cases):
    print(f"\n{label}")
    for key, index, meaning in [
        ("t43_ref", t43_index, "normalized features times gamma"),
        ("t44_ref", t44_index, "LayerNorm output after beta"),
        ("t45_ref", t45_index, "INT8 projection input"),
    ]:
        values = np.concatenate([c[key].reshape(-1) for c in cases])
        print(
            f"  T{index} ({meaning}) range:",
            f"{values.min()} to {values.max()}",
        )


report_affine_reference("Benchmark", affine_reference_cases[:1])
report_affine_reference("Calibration", affine_reference_cases[1:])


T6 (learned scale gamma)
  Shape: (128,)
  Stored dtype: float32
  Minimum: 0.922644913
  Maximum: 1.139880538
  Maximum absolute value: 1.139880538
  First 8 values: [0.96397287 1.0049611  0.9837676  0.9226449  0.96249086 0.98930055
 1.0272598  0.9648354 ]

T7 (learned bias beta)
  Shape: (128,)
  Stored dtype: float32
  Minimum: -0.242233545
  Maximum: 0.219664216
  Maximum absolute value: 0.242233545
  First 8 values: [-0.0593875   0.0696523   0.03833386 -0.03081142  0.04557584 -0.1962216
  0.11669759 -0.07880756]

T45 (INT8 projection input)
  Shape: (np.int32(1), np.int32(32), np.int32(128))
  Scale: 0.03362198546528816
  Zero point: -17
  Representable real range: -3.732040386646986 to 4.841565907001495

Benchmark
  T43 (normalized features times gamma) range: -3.614773750305176 to 2.9820332527160645
  T44 (LayerNorm output after beta) range: -3.7485108375549316 to 3.0682313442230225
  T45 (INT8 projection input) range: -128 to 74

Calibration
  T43 (normalized features times ga

In [18]:
# Op 12–14: learned affine transform and INT8 quantization.

GAMMA_FRAC_BITS = 16
BETA_FRAC_BITS = 16
GAMMA_SCALE = 2.0 ** (-GAMMA_FRAC_BITS)
BETA_SCALE = 2.0 ** (-BETA_FRAC_BITS)

int32_limits = np.iinfo(np.int32)
int64_max = int(np.iinfo(np.int64).max)

# Offline conversion of learned FP32 constants.
def quantize_parameter(values, scale):
    rounded = np.rint(values.astype(np.float64) / scale)
    assert rounded.min() >= int32_limits.min
    assert rounded.max() <= int32_limits.max
    return rounded.astype(np.int32)


ln1_gamma_q = quantize_parameter(ln1_gamma_fp32, GAMMA_SCALE)
ln1_beta_q = quantize_parameter(ln1_beta_fp32, BETA_SCALE)

gamma_approx = ln1_gamma_q.astype(np.float64) * GAMMA_SCALE
beta_approx = ln1_beta_q.astype(np.float64) * BETA_SCALE

T43_FRAC_BITS = T42_FRAC_BITS + GAMMA_FRAC_BITS
T43_SCALE = 2.0 ** (-T43_FRAC_BITS)
T44_SCALE = T43_SCALE

BETA_ALIGN_SHIFT = T43_FRAC_BITS - BETA_FRAC_BITS
assert BETA_ALIGN_SHIFT >= 0

# Offline conversion of the final requantization multiplier.
T45_REQUANT_SHIFT = 47
T45_REAL_MULTIPLIER = T44_SCALE / T45_SCALE
T45_MULTIPLIER_Q = int(np.rint(
    T45_REAL_MULTIPLIER * (1 << T45_REQUANT_SHIFT)
))
assert 0 < T45_MULTIPLIER_Q <= int64_max

T45_ROUNDING_OFFSET = 1 << (T45_REQUANT_SHIFT - 1)
multiplier_approx = T45_MULTIPLIER_Q * 2.0 ** (-T45_REQUANT_SHIFT)

print("T6 (gamma): INT32, scale =", GAMMA_SCALE)
print("T7 (beta): INT32, scale =", BETA_SCALE)
print("Gamma max quantization error:",
      f"{np.abs(gamma_approx - ln1_gamma_fp32).max():.9e}")
print("Beta max quantization error:",
      f"{np.abs(beta_approx - ln1_beta_fp32).max():.9e}")
print("T43/T44 (affine intermediates): INT64, scale =", T44_SCALE)
print("Beta alignment multiplier:", 1 << BETA_ALIGN_SHIFT)
print("T45 multiplier integer:", T45_MULTIPLIER_Q)
print("T45 right shift:", T45_REQUANT_SHIFT)
print("T45 scale:", T45_SCALE)
print("T45 zero point:", T45_ZERO_POINT)
print(
    "Multiplier relative error:",
    f"{abs(multiplier_approx / T45_REAL_MULTIPLIER - 1):.9e}",
)

ln1_affine_integer_trial = []
assert len(normalization_integer_trial) == len(affine_reference_cases)

for normalized, reference in zip(
    normalization_integer_trial, affine_reference_cases
):
    assert normalized["name"] == reference["name"]

    # Op 12: integer multiplication.
    # INT32 operands are promoted before multiplication.
    t43_q = (
        normalized["t42_q"].astype(np.int64)
        * ln1_gamma_q.astype(np.int64)
    )

    # Op 13: align beta to the product scale, then add.
    aligned_beta_q = (
        ln1_beta_q.astype(np.int64) * (1 << BETA_ALIGN_SHIFT)
    )

    # Conservative bound before the addition.
    product_bound = max(abs(int(t43_q.min())), abs(int(t43_q.max())))
    beta_bound = max(
        abs(int(aligned_beta_q.min())),
        abs(int(aligned_beta_q.max())),
    )
    assert product_bound + beta_bound <= int64_max

    t44_q = t43_q + aligned_beta_q

    # Check multiplication plus rounding before calculating it.
    affine_bound = max(abs(int(t44_q.min())), abs(int(t44_q.max())))
    assert (
        affine_bound * T45_MULTIPLIER_Q + T45_ROUNDING_OFFSET
        <= int64_max
    )

    # Op 14: integer-only candidate requantization.
    # Nearest rounding, ties toward positive infinity.
    rounded_q = (
        t44_q * T45_MULTIPLIER_Q + T45_ROUNDING_OFFSET
    ) >> T45_REQUANT_SHIFT

    before_saturation = rounded_q + T45_ZERO_POINT
    saturation_count = int(np.count_nonzero(
        (before_saturation < -128) | (before_saturation > 127)
    ))
    t45_q = np.clip(before_saturation, -128, 127).astype(np.int8)

    # Floating-point calculations below are diagnostics only.
    t43_approx = t43_q.astype(np.float64) * T43_SCALE
    t44_approx = t44_q.astype(np.float64) * T44_SCALE

    # Isolate parameter quantization from earlier normalization errors.
    ideal_affine_from_candidate = (
        normalized["t42_approx"] * ln1_gamma_fp32 + ln1_beta_fp32
    )
    parameter_error = np.abs(
        t44_approx - ideal_affine_from_candidate
    )

    # Compare fixed-point requantization against ideal quantization
    # of OUR represented T44, using the SAME rounding rule.
    ideal_t45_q = np.clip(
        np.floor(t44_approx / T45_SCALE + 0.5) + T45_ZERO_POINT,
        -128, 127,
    ).astype(np.int8)

    # Cast before subtraction to avoid INT8 overflow.
    t45_difference = (
        t45_q.astype(np.int32)
        - reference["t45_ref"].astype(np.int32)
    )

    ln1_affine_integer_trial.append({
        "name": normalized["name"],
        "t43_q": t43_q,
        "t44_q": t44_q,
        "t45_q": t45_q,
        "t43_approx": t43_approx,
        "t44_approx": t44_approx,
        "t43_error": np.abs(t43_approx - reference["t43_ref"]),
        "t44_error": np.abs(t44_approx - reference["t44_ref"]),
        "parameter_error": parameter_error,
        "t45_ref": reference["t45_ref"],
        "t45_difference": t45_difference,
        "requant_mismatch_count": int(np.count_nonzero(
            t45_q != ideal_t45_q
        )),
        "saturation_count": saturation_count,
    })


def report_affine_integer(label, trials):
    print(f"\n{label}")

    for key, meaning in [
        ("t43_error", "T43 (normalized features times gamma) vs FP32"),
        ("t44_error", "T44 (affine LayerNorm output) vs FP32"),
        ("parameter_error", "Additional error from gamma/beta conversion"),
    ]:
        errors = np.concatenate([t[key].reshape(-1) for t in trials])
        print(f"  {meaning}:")
        print(f"    Max absolute error:  {errors.max():.9e}")
        print(f"    Mean absolute error: {errors.mean():.9e}")

    differences = np.concatenate([
        t["t45_difference"].reshape(-1) for t in trials
    ])
    values = np.concatenate([t["t45_q"].reshape(-1) for t in trials])
    matched = int(np.count_nonzero(differences == 0))

    print(f"  T45 (candidate INT8) range: {values.min()} to {values.max()}")
    print(f"  T45 exact matches: {matched}/{differences.size}")
    print(f"  T45 match percentage: {100 * matched / differences.size:.3f}%")
    print("  T45 maximum integer difference:",
          int(np.abs(differences).max()))

    unique, counts = np.unique(differences, return_counts=True)
    print("  Signed difference counts:", {
        int(d): int(c) for d, c in zip(unique, counts)
    })

    print("  Fixed multiplier vs ideal quantization mismatches:",
          sum(t["requant_mismatch_count"] for t in trials))
    print("  Candidate values requiring saturation:",
          sum(t["saturation_count"] for t in trials))
    print("  Intermediate width checks: pass")


report_affine_integer("Benchmark", ln1_affine_integer_trial[:1])
report_affine_integer("Calibration", ln1_affine_integer_trial[1:])

first = ln1_affine_integer_trial[0]
location = (0, 0, 0)

print("\nFirst benchmark token, first feature:")
print("  T6 (gamma), stored integer:", ln1_gamma_q[0])
print("  T7 (beta), stored integer:", ln1_beta_q[0])
print("  T43 (scaled normalized feature), integer:", first["t43_q"][location])
print("  T44 (affine output), integer:", first["t44_q"][location])
print("  T44 (represented value):", f'{first["t44_approx"][location]:.9f}')
print("  T45 (candidate INT8):", first["t45_q"][location])
print("  T45 (TFLite reference INT8):", first["t45_ref"][location])

T6 (gamma): INT32, scale = 1.52587890625e-05
T7 (beta): INT32, scale = 1.52587890625e-05
Gamma max quantization error: 7.629394531e-06
Beta max quantization error: 7.601454854e-06
T43/T44 (affine intermediates): INT64, scale = 2.3283064365386963e-10
Beta alignment multiplier: 65536
T45 multiplier integer: 974600
T45 right shift: 47
T45 scale: 0.03362198546528816
T45 zero point: -17
Multiplier relative error: 3.956765795e-07

Benchmark
  T43 (normalized features times gamma) vs FP32:
    Max absolute error:  2.628150489e-02
    Mean absolute error: 1.019115742e-02
  T44 (affine LayerNorm output) vs FP32:
    Max absolute error:  2.627590206e-02
    Mean absolute error: 1.019107212e-02
  Additional error from gamma/beta conversion:
    Max absolute error:  2.371183291e-05
    Mean absolute error: 5.079321407e-06
  T45 (candidate INT8) range: -128 to 74
  T45 exact matches: 2854/4096
  T45 match percentage: 69.678%
  T45 maximum integer difference: 1
  Signed difference counts: {-1: 624, 

### Block-0 first LayerNorm: integer candidate, Op 0–14

Source block-0 SHA-256:
`06481f6c5151a83e3c01896d314d902ee9608773d94834848777ccc7282d87a9`

Test inputs: one 32-token benchmark sequence and seven calibration
sequences; hidden size 128.

| Tensor | Meaning | Candidate representation |
|---|---|---|
| T0 | Block input | INT8, scale 2^-8, zero point 0 |
| T31/T32 | Per-token means / broadcast means | INT32, scale 2^-15 |
| T33 | Squared differences | INT64 at 2^-30; requantized to INT32 at 2^-15 |
| T34 | Variance | INT32, scale 2^-22 |
| T35 | Variance plus epsilon | INT32, scale 2^-22; epsilon integer 42 |
| T36 | Inverse standard deviation | INT32, scale 2^-16 |
| T42 | Normalized features | INT64 at 2^-31; requantized to INT32 at 2^-16 |
| T6/T7 | Learned gamma / beta | INT32, scale 2^-16 |
| T43/T44 | Affine intermediates / LayerNorm output | INT64, scale 2^-32 |
| T45 | Projection input | INT8, scale 0.03362198546528816, zero point -17 |

Op 5 uses integer division followed by integer square root:
`q_T36 = isqrt(2^54 // q_T35)`, with floor rounding.

Final T45 requantization uses multiplier 974600, right shift 47,
nearest rounding with ties toward positive infinity, zero-point
addition, and INT8 saturation.

| Result | Benchmark | Calibration |
|---|---:|---:|
| T44 maximum absolute error vs FP32 | 0.0262759 | 0.0281122 |
| T44 mean absolute error vs FP32 | 0.0101911 | 0.0101453 |
| T45 exact matches | 2854/4096 | 20020/28672 |
| Maximum T45 integer difference | 1 | 1 |
| Fixed multiplier vs ideal quantization mismatches | 0 | 0 |
| Candidate saturation count | 0 | 1 |

All implemented width checks passed for the tested inputs.
Earlier input/normalization errors dominate; parameter conversion
and output requantization add comparatively little error.

This is a Python-validated integer candidate. C, Spike, RTL,
complete-block accuracy, and performance have not yet been validated
for this LayerNorm.